# Gestructureerde data: preprocessing dataset

In dit voorbeeld gaan we verder werken op de titanic dataset bestudeerd in voorgaande notebook.
Hierbij gaan we preprocessing toevoegen of in meer detail bespreken.

Hierbij is een belangrijk onderscheid tussen pytorch en tensorflow op te merken. Hierbij is het belangrijk om te realiseren dat pytorch verwacht dat alle data dat uit de __get__item functie komt numeriek is. Deze moet niet noodzakelijk reeds geschaald zijn maar moet wel dat datatype hebben. Het is dus belangrijk om een goede keuze te maken op welke plaats de nodige preprocessing stappen uitgevoerd worden. 
In het geval van tensorflow (in het geval van mixed datatypes) plaatsen we van in het begin elke feature apart. Daarna voegen we gelijkaardige features samen om deze te preprocessen. Pas op het einde van dit proces worden alle features samengevoegd tot een numerieke tensor.

### Pytorch

In [9]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_features = ['age', 'n_siblings_spouses', 'parch', 'fare']
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['sex', 'class', 'embark_town', 'deck', 'alone']
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Load Titanic dataset
titanic = pd.read_csv("https://storage.googleapis.com/tf-datasets/titanic/train.csv")

class TitanicDataset(Dataset):
    def __init__(self, dataframe, transform, fit=False):
        if fit is True:
            transform.fit(dataframe)

        self.transform = transform
        self.dataframe = transform.transform(dataframe)

        print(self.dataframe)
        
        X=self.dataframe[1:]
        y=self.dataframe[0] # label kolom selecteren

        print(X)

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
     
    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# splits in train test
titanic_train = titanic.sample(frac=0.8)
titanic_test = titanic.drop(titanic_train.index)

# make datasets, vermijd dataleakage
train_dataset = TitanicDataset(titanic_train, preprocessor, fit=True)
test_dataset = TitanicDataset(titanic_test, train_dataset.transform)

# dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

[[-1.03095939  0.43217738 -0.46767929 ...  0.          1.
   0.        ]
 [-0.12508657  1.35538181 -0.46767929 ...  1.          1.
   0.        ]
 [ 0.36902588  0.43217738 -0.46767929 ...  1.          1.
   0.        ]
 ...
 [-0.61919902 -0.49102706 -0.46767929 ...  1.          0.
   1.        ]
 [-0.04273449  0.43217738 -0.46767929 ...  1.          1.
   0.        ]
 [-0.37214279 -0.49102706 -0.46767929 ...  1.          0.
   1.        ]]
[[-0.12508657  1.35538181 -0.46767929 ...  1.          1.
   0.        ]
 [ 0.36902588  0.43217738 -0.46767929 ...  1.          1.
   0.        ]
 [-1.85448014 -0.49102706  2.05678313 ...  1.          1.
   0.        ]
 ...
 [-0.61919902 -0.49102706 -0.46767929 ...  1.          0.
   1.        ]
 [-0.04273449  0.43217738 -0.46767929 ...  1.          1.
   0.        ]
 [-0.37214279 -0.49102706 -0.46767929 ...  1.          0.
   1.        ]]
[[-0.12508657 -0.49102706 -0.46767929 ...  1.          0.
   1.        ]
 [-1.27801561 -0.49102706 -0.46767929 .

## Oefening: Inkomensdataset

Met onderstaande code downloaden we een csv met gegevens rond het inkomen van volwassenen.
Plaats in de code-cellen eronder de nodige code om de volgende preprocessingstappen uit te voeren met pytorch:

* Splits in train-test
* Label kolom is het inkomen -> voer hiervoor ordinal encoding uit
* Voer normalisatie uit van de numerieke kolommen zodat alle waarden tussen 0 en 1 liggen (geen standaardverdeling)
* Voer ordinal encoding uit op de occupation en native_country kolom
* Voer one-hot encoding uit op de overige niet-numerieke kolommen